In [87]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re

In [88]:
def convert_to_gigabytes(size_str):
    # Handle empty/null values
    if pd.isna(size_str):
        return None
    match = re.search(r"([\d\.]+)\s*([a-zA-Z]+)", str(size_str).strip())

    if not match:
        return None  # Return None if the format doesn't match

    value = float(match.group(1))
    unit = match.group(2).upper()

    # Define conversion multipliers (using the standard 1024 binary prefix)
    multipliers = {"B": 1, "KB": 1024, "MB": 1024**2, "GB": 1024**3, "TB": 1024**4}

    # Handle shorthand units (e.g., 'K' instead of 'KB')
    if unit in ["K", "M", "G", "T"]:
        unit += "B"

    # Multiply the value by the correct unit multiplier
    multiplier = multipliers.get(unit, 1)

    return int(value * multiplier) / 1024**3


def time_string_to_hours(time_str):
    try:
        # Convert to string and strip any accidental whitespace
        time_str = str(time_str).strip()

        # Check if the "Days-" portion exists
        if "-" in time_str:
            days_str, time_part = time_str.split("-")
            days = int(days_str)
        else:
            days = 0
            time_part = time_str  # The whole string is just HH:MM:SS

        # Split the time part into hours, minutes, and seconds
        hours, minutes, seconds = map(int, time_part.split(":"))

        # Calculate total hours
        return (days * 24) + hours + (minutes / 60) + (seconds / 3600)

    except (ValueError, AttributeError):
        # Return None (which pandas converts to NaN) for bad data
        return None

In [89]:

df = pd.read_csv('jobs_history.csv', sep='|', encoding='utf-8')

# Display the first 5 rows to make sure it loaded correctly
df.head()

,JobID,User,Partition,AllocCPUS,AllocTRES,ReqMem,MaxRSS,Elapsed,Timelimit,State,Start,End,WorkDir
0,985911_195,bsobieski,hopper,4,"billing=151,cpu=4,gres/gpu:h100=1,gres/gpu=1,m...",40G,NaN,00:00:00,10:00:00,PENDING,Unknown,Unknown,/mnt/evafs/faculty/home/bsobieski/pdb
1,985911_196,bsobieski,hopper,4,"billing=151,cpu=4,gres/gpu:h100=1,gres/gpu=1,m...",40G,NaN,00:00:00,10:00:00,PENDING,Unknown,Unknown,/mnt/evafs/faculty/home/bsobieski/pdb
2,985911_197,bsobieski,hopper,4,"billing=151,cpu=4,gres/gpu:h100=1,gres/gpu=1,m...",40G,NaN,00:00:00,10:00:00,PENDING,Unknown,Unknown,/mnt/evafs/faculty/home/bsobieski/pdb
3,985911_198,bsobieski,hopper,4,"billing=151,cpu=4,gres/gpu:h100=1,gres/gpu=1,m...",40G,NaN,00:00:00,10:00:00,PENDING,Unknown,Unknown,/mnt/evafs/faculty/home/bsobieski/pdb
4,985911_199,bsobieski,hopper,4,"billing=151,cpu=4,gres/gpu:h100=1,gres/gpu=1,m...",40G,NaN,00:00:00,10:00:00,PENDING,Unknown,Unknown,/mnt/evafs/faculty/home/bsobieski/pdb


In [90]:
df["ReqMem"] = df["ReqMem"].apply(convert_to_gigabytes)
df["MaxRSS"] = df["MaxRSS"].apply(convert_to_gigabytes)
df["Timelimit"] = df["Timelimit"].apply(time_string_to_hours)
df["Elapsed"] = df["Elapsed"].apply(time_string_to_hours)
df["ReqMem"] = df["ReqMem"].fillna(0)
df["MaxRSS"] = df["MaxRSS"].fillna(0)
df.head()

,JobID,User,Partition,AllocCPUS,AllocTRES,ReqMem,MaxRSS,Elapsed,Timelimit,State,Start,End,WorkDir
0,985911_195,bsobieski,hopper,4,"billing=151,cpu=4,gres/gpu:h100=1,gres/gpu=1,m...",40.0,0.0,0.0,10.0,PENDING,Unknown,Unknown,/mnt/evafs/faculty/home/bsobieski/pdb
1,985911_196,bsobieski,hopper,4,"billing=151,cpu=4,gres/gpu:h100=1,gres/gpu=1,m...",40.0,0.0,0.0,10.0,PENDING,Unknown,Unknown,/mnt/evafs/faculty/home/bsobieski/pdb
2,985911_197,bsobieski,hopper,4,"billing=151,cpu=4,gres/gpu:h100=1,gres/gpu=1,m...",40.0,0.0,0.0,10.0,PENDING,Unknown,Unknown,/mnt/evafs/faculty/home/bsobieski/pdb
3,985911_198,bsobieski,hopper,4,"billing=151,cpu=4,gres/gpu:h100=1,gres/gpu=1,m...",40.0,0.0,0.0,10.0,PENDING,Unknown,Unknown,/mnt/evafs/faculty/home/bsobieski/pdb
4,985911_199,bsobieski,hopper,4,"billing=151,cpu=4,gres/gpu:h100=1,gres/gpu=1,m...",40.0,0.0,0.0,10.0,PENDING,Unknown,Unknown,/mnt/evafs/faculty/home/bsobieski/pdb


In [91]:
from pandas import DataFrame


def extract_key_values(text):
    if pd.isna(text): # Handle empty rows
        return {}
    
    # Split by ',' to get pairs, then split by '=' to separate keys and values
    # The '1' in split('=', 1) ensures it only splits on the first equals sign
    return dict(item.split('=', 1) for item in str(text).split(','))

# 3. Apply the function and turn the resulting dictionaries into columns
expanded_df = df['AllocTRES'].apply(extract_key_values).apply(pd.Series)

# 4. Join the new columns back to your original DataFrame
df: DataFrame = pd.concat([df, expanded_df], axis=1)

df = df.drop(['WorkDir','billing','AllocTRES', 'Start', 'End'], axis=1)

df

,JobID,User,Partition,AllocCPUS,ReqMem,MaxRSS,Elapsed,Timelimit,State,cpu,gres/gpu:h100,gres/gpu,mem,node,gres/gpu:a100,gres/gpu:h200,gres/gpu:rtx6000,gres/gpu:h200_4g.71gb
0,985911_195,bsobieski,hopper,4,40.000000,0.000000,0.000000,10.0,PENDING,4,1,1,40G,1,NaN,NaN,NaN,NaN
1,985911_196,bsobieski,hopper,4,40.000000,0.000000,0.000000,10.0,PENDING,4,1,1,40G,1,NaN,NaN,NaN,NaN
2,985911_197,bsobieski,hopper,4,40.000000,0.000000,0.000000,10.0,PENDING,4,1,1,40G,1,NaN,NaN,NaN,NaN
3,985911_198,bsobieski,hopper,4,40.000000,0.000000,0.000000,10.0,PENDING,4,1,1,40G,1,NaN,NaN,NaN,NaN
4,985911_199,bsobieski,hopper,4,40.000000,0.000000,0.000000,10.0,PENDING,4,1,1,40G,1,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
122,1733665.batch,NaN,NaN,124,0.000000,394.699566,0.403611,NaN,CANCELLED,124,4,4,500G,1,NaN,NaN,NaN,NaN
123,1733669,pbugyi,short,1,24.414062,0.000000,0.454722,8.0,RUNNING,1,NaN,NaN,25000M,1,NaN,NaN,NaN,NaN
124,1733669.batch,NaN,NaN,1,0.000000,0.000000,0.454722,NaN,RUNNING,1,NaN,NaN,25000M,1,NaN,NaN,NaN,NaN
125,1733670,mgromadzki,hopper,124,500.000000,0.000000,0.213889,14.0,RUNNING,124,4,4,500G,1,NaN,NaN,NaN,NaN


In [92]:
df['Time_Efficiency'] = np.where(df['Timelimit'] > 0, 
                                 df['Elapsed'] / df['Timelimit'], 
                                 np.nan)

under_50_pct = (df['Time_Efficiency'] < 0.5).mean() * 100
print(df[['JobID', 'Elapsed', 'Timelimit', 'Time_Efficiency']])
print(f"Procent zadań ze wskaźnikiem < 0.5: {under_50_pct:.2f}%\n")

             JobID   Elapsed  Timelimit  Time_Efficiency
0       985911_195  0.000000       10.0         0.000000
1       985911_196  0.000000       10.0         0.000000
2       985911_197  0.000000       10.0         0.000000
3       985911_198  0.000000       10.0         0.000000
4       985911_199  0.000000       10.0         0.000000
..             ...       ...        ...              ...
122  1733665.batch  0.403611        NaN              NaN
123        1733669  0.454722        8.0         0.056840
124  1733669.batch  0.454722        NaN              NaN
125        1733670  0.213889       14.0         0.015278
126  1733670.batch  0.213889        NaN              NaN

[127 rows x 4 columns]
Procent zadań ze wskaźnikiem < 0.5: 50.39%



In [93]:
short_tasks_long_res = df[(df['Timelimit'] > 72) & (df['Elapsed'] < 8)]

print(short_tasks_long_res[['JobID', 'Elapsed', 'Timelimit']])
print("\n")

      JobID  Elapsed  Timelimit
43  1733597      0.0      120.0




In [ ]:


df['Wasted_Mem_GB'] = df['ReqMem'] - df['MaxRSS']
df['Mem_Utilization_Pct'] = (df['MaxRSS'] / df['ReqMem']) * 100

lowMemory = df[(df['Mem_Utilization_Pct'] < 40) & (df['MaxRSS'] != None)]

print(lowMemory[['JobID', 'ReqMem', 'MaxRSS', 'Wasted_Mem_GB', 'Mem_Utilization_Pct']])
print("\n")

          JobID      ReqMem  MaxRSS  Wasted_Mem_GB  Mem_Utilization_Pct
0    985911_195   40.000000     0.0      40.000000                  0.0
1    985911_196   40.000000     0.0      40.000000                  0.0
2    985911_197   40.000000     0.0      40.000000                  0.0
3    985911_198   40.000000     0.0      40.000000                  0.0
4    985911_199   40.000000     0.0      40.000000                  0.0
..          ...         ...     ...            ...                  ...
117     1733663   24.414062     0.0      24.414062                  0.0
119     1733664   24.414062     0.0      24.414062                  0.0
121     1733665  500.000000     0.0     500.000000                  0.0
123     1733669   24.414062     0.0      24.414062                  0.0
125     1733670  500.000000     0.0     500.000000                  0.0

[75 rows x 5 columns]




In [ ]:
#Graphs to add